# 06 · Decoder & submission

**Tujuan.** Menggabungkan keputusan bola hidup/mati dan keputusan tim menjadi satu label per frame, menulis file
submission, memvalidasinya dengan validator resmi, dan mencatat ringkasannya.

| | |
|---|---|
| **Input** | `cache/prep.npz`, `cache/dead.npz` |
| **Output** | `submissions/<nama>.csv`, laporan `cache/reports/06_decoder_submission.*` |
| **Waktu** | ± 30 s (+ bootstrap) |

### Versi saat ini: **v0** — submission end-to-end pertama (Fase 1.4)

Tujuannya **bukan** skor terbaik, melainkan pipeline lengkap yang valid dan bisa direproduksi, sebagai titik awal
yang jujur untuk dibandingkan dengan versi berikutnya.

- `D` = keluaran `03_bola_mati` apa adanya.
- `H`/`A` di frame hidup = **Viterbi 3 keadaan** (H, A, D) di atas satu bukti saja: tim pemain terdekat ke bola.
  Setiap frame dengan bola terlihat menyumbang +1 untuk tim terdekat; frame tanpa bola tidak menyumbang apa-apa,
  sehingga decoder mempertahankan possession terakhir (sesuai CHALLENGE.md: saat bola lepas/diperebutkan,
  possession tetap milik tim terakhir).
- Berganti H↔A "membayar" `SWITCH_HA`: tim baru harus unggul ± `SWITCH_HA` frame-bukti bersih. Masuk/keluar `D`
  gratis karena `D` sudah diputuskan oleh tahap 03.

Bandingkan dengan baseline resmi: voting mayoritas ± 50 frame tanpa konsep bola mati selain "bola hilang".

**Versi berikutnya (Fase 2)** mengganti bukti tunggal ini dengan probabilitas dari `05_model_possession`
(bentuk tim + pemain terdekat) dan aturan restart.

In [ ]:
%run ./common.ipynb

## Parameter

| Parameter | Nilai | Alasan |
|---|---|---|
| `VERSION` | `v0` | dicatat di nama file dan log eksperimen |
| `SWITCH_HA` | 25 | ± 1 s bukti bersih untuk mengganti possession — selaras dengan ide "kontrol": sentuhan sesaat tidak cukup. Tidak disetel di GT publik |
| `BOOTSTRAP_N` | 1000 | interval bootstrap resmi untuk skor publik (EVALUATION.md) |

In [ ]:
VERSION = "v0"
SWITCH_HA = 25.0
BOOTSTRAP_N = 1000
SUBMISSION_NAME = "{}_{}".format(TEAM_NAME, VERSION)

In [ ]:
rep = Report("06_decoder_submission")
z = load_prep()
with np.load(CACHE / "dead.npz") as dz:
    d_mask, near = dz["d_mask"], dz["near_team"]
rep("versi {} | D dari 03: {:.3f} frame dinilai | bola terlihat di frame hidup: {:.1f}%".format(
    VERSION, d_mask[GRADED].mean(), 100 * (near[GRADED & ~d_mask] != 0).mean()))

## 1. Emisi dan decoder

Keadaan: 0 = `H`, 1 = `A`, 2 = `D`.

| Keadaan | Emisi di frame hidup (`d_mask` = 0) | Emisi di frame `D` |
|---|---|---|
| `H` | +½ bila H terdekat, −½ bila A terdekat, 0 bila bola hilang | −10⁴ |
| `A` | kebalikannya | −10⁴ |
| `D` | −10⁴ | +10⁴ |

Selisih emisi H − A per frame = ±1, jadi `SWITCH_HA` terbaca langsung sebagai "jumlah frame-bukti bersih".
Decoder dijalankan per babak; frame di luar babak diberi `X` (tidak dinilai).

In [ ]:
BIG = 1e4
emis = np.zeros((N_FRAMES, 3))
emis[:, 0] = 0.5 * near
emis[:, 1] = -0.5 * near
emis[d_mask, 0] = emis[d_mask, 1] = -BIG
emis[d_mask, 2] = BIG
emis[~d_mask, 2] = -BIG

TRANS = np.array([[0.0, SWITCH_HA, 0.0],
                  [SWITCH_HA, 0.0, 0.0],
                  [0.0, 0.0, 0.0]])

labels = np.full(N_FRAMES, "X", dtype="<U1")
with timer("viterbi 3 keadaan"):
    for h, (a, b) in HALVES.items():
        path = viterbi(emis[a:b + 1], TRANS)
        labels[a:b + 1] = np.array(["H", "A", "D"])[path]

## 2. Tulis & validasi

`write_submission` menulis dengan format tetap dan mengembalikan SHA-256 — hash inilah yang nanti dibandingkan
saat uji reproduksi. `validate_file` memakai validator resmi.

In [ ]:
path, digest = write_submission(labels, SUBMISSION_NAME)
ok, summary = validate_file(path)
rep("file: {} | sha256 {}".format(path.relative_to(ROOT), digest))
rep("validasi resmi: {} | {}".format("LULUS" if ok else "GAGAL", summary))
assert ok, summary

graded_lab = labels[GRADED]
alive = np.isin(graded_lab, ["H", "A"])
rep("seluruh match: H {:.1f}% | A {:.1f}% | D {:.1f}% | porsi H dari waktu hidup {:.1f}%".format(
    100 * (graded_lab == "H").mean(), 100 * (graded_lab == "A").mean(), 100 * (graded_lab == "D").mean(),
    100 * (graded_lab[alive] == "H").mean()))

## 3. Cek kebenaran pada GT publik (dengan interval bootstrap)

Dibandingkan dengan baseline resmi pada jendela yang sama. Ingat: satu possession ≈ ±6 poin spellF1 di sini;
yang dicari adalah *tidak ada yang rusak* dan arah yang masuk akal, bukan selisih kecil.

In [ ]:
cols = ["spell_f1_mean", "spell_bootstrap_low", "spell_bootstrap_high", "macro_f1", "n_pred_spells",
        "n_gt_spells", "mean_spell_duration_s", "alive_dead_f1", "team_acc_gt_alive"]
board = {}
with timer("skor publik + bootstrap"):
    board[SUBMISSION_NAME] = score_public(path, bootstrap_n=BOOTSTRAP_N)
    board["baseline_smooth50 (resmi)"] = score_public(
        OFFICIAL / "evaluation" / "results" / "baseline_smooth50.csv", bootstrap_n=BOOTSTRAP_N)
board_df = pd.DataFrame(board).T[cols].astype(float).round(4)
rep.table(board_df, "GT publik:")

In [ ]:
rep.save({"version": VERSION, "file": str(path.relative_to(ROOT)), "sha256": digest, "summary": summary,
          "public": board_df.reset_index().to_dict("records"),
          "params": {"SWITCH_HA": SWITCH_HA}})

## Ringkasan & keputusan (v0)

| GT publik | v0 | baseline resmi `smooth50` |
|---|---|---|
| spellF1 [interval bootstrap 5–95%] | ± 0,34 [0,19 – 0,50] | ± 0,33 [0,21 – 0,45] |
| macroF1 | ± 0,72 | ± 0,62 |
| possession dilaporkan (GT 17) | 19 | 26 |
| akurasi tim pada frame hidup GT | **± 0,69** | ± 0,83 |
| F1 hidup/mati | ± 0,88 | ± 0,90 |

Seluruh match: ± 300 possession dengan rata-rata ± 11 s; porsi `D` ± 0,48.

**Bacaan**

1. **Gerbang Fase 1 terpenuhi:** pipeline 00 → 06 menghasilkan file valid dari data mentah dengan satu perintah.
2. **spellF1 publik tidak bisa dibedakan dari baseline** — intervalnya hampir sama lebar dengan nilainya. Ini titik
   acuan, bukan kemenangan.
3. **Hambatan utama adalah presisi `D`, bukan H/A.** Akurasi tim pada frame hidup GT turun ke ± 0,69 karena `D`
   palsu dari tahap 03 "memakan" frame hidup (presisi `D` publik ± 0,52), dan setiap `D` palsu memecah possession.
   Jumlah possession yang mendekati GT (19 vs 17) sebagian kebetulan: `D` palsu menambah possession, decoder yang
   enggan berganti menguranginya.

**Keputusan**

- v0 menjadi acuan untuk semua versi berikutnya (`docs/experiment_log.md` #1).
- **Perbaikan presisi `D` (Fase 3.1) dimajukan** berdampingan dengan Fase 2: tanpa itu, model H/A sebaik apa pun
  tetap kehilangan frame dan possession yang "dicuri" `D` palsu.
- Fase 2 mengganti bukti tunggal "pemain terdekat" dengan model yang juga memakai bentuk tim (temuan 02 #4).